# 04 - Binary relevance y clasificación multietiqueta

Se reserva un 20% real como test multietiqueta. Con el 80% restante se entrena un clasificador one-vs-all por sistema cristalino (balanceado con SMOTE), y se evalúa en el test real.

In [ ]:
from csp_perovskites import config, io, plotting, preprocessing as pp
from csp_perovskites import datasets as ds, evaluation as ev, models

## Parámetros

In [ ]:
SCALER = "minmax"
TUNE = False      # True = GridSearchCV por sistema cristalino (mucho más lento)
MODELS = None     # None = los 12 modelos
N_JOBS = config.N_JOBS  # -1 = todos los núcleos; bajarlo si falta RAM; 1 = secuencial

df = pp.clean_master(io.load_master())
df_encoded = pp.encode_multilabel(df)
binary = ds.make_binary_splits(df_encoded, seed=config.RANDOM_STATE)
print("Etiquetas por compuesto en el test:", pp.label_cardinality(binary.Y_test).to_dict())

## Clasificación binaria por sistema cristalino

In [ ]:
accuracy, predictions = ev.evaluate_binary(binary, models.get_classifiers(MODELS), scaler=SCALER, tune=TUNE,
                                           n_jobs=N_JOBS)
io.save_table(accuracy, "binary_accuracy.csv", index=True)

io.save_fig(plotting.accuracy_heatmap(accuracy), "crystal_system_accuracy.png")
io.save_fig(plotting.average_accuracy_bar(accuracy), "average_model_performance.png")
accuracy.round(2)

## Métricas multietiqueta (Extra Trees, LightGBM, Random Forest)

In [ ]:
report = ev.multilabel_report(binary.Y_test, predictions, config.TOP_MODELS)
io.save_table(report, "multilabel_metrics.csv", index=True)
report.round(4)